In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
import numpy as np
from pathlib import Path

%matplotlib inline

**Jakie zastosowanie może mieć model klasyfikujący gatunki muzyczne?**

Wyobraźmy sobie system rekomendujący użytkownikom piosenki (np. Spotify). Wiedza o tym do jakiego gatunku należy dana piosenka może się przydać
jeśli chcemy zarekomendować użytkownikom nowe piosenki z jakiegoś gatunku, albo po prostu jeśli chcemy mieć statystyki odnośnie tego co jest teraz popularne
(w oczywisty sposób manualne kategoryzowanie nie wchodzi w rachubę).

**Dane i cel projetku**

Pracujemy na datasecie GTZAN (1000 piosenek po 30 sekund, 10 gatunków). Danymi do treningu są dane numeryczne wyekstraktowane z surowych plików audio.

Celem projektu jest oczywiście stworzenie jak najlepszego modelu do klasyfikacji gatunków muzycznych, ale również nauka o pracy z audio w Machine Learningu.

**Preprocessing**

Do ekstrakcji cech z plików audio używaliśmy biblioteki `librosa`, która posiada wszystkie potrzebne funkcje do tego celu. 
W oczywisty sposób nie mieliśmy przez to żadnych problemów z brakującymi wartościami. Wszystkie cechy były cechami numerycznymi, w większości są to średnie i wariancje różnych statystyk, które są popularne w analizie audio.

Chcieliśmy znaleźć sposób na rozdzielenie podobnych gatunków (np. rock i blues). Jedną z cech, które wymyślilśmy jest `dist_from_genre`, gdzie dla każdego utworu obliczamy dystans DTW (według jednej z cech) od typowego utworu z danego gatunku. Niestety nie pomogło to w klasyfikacji.

**Modele**

Klasyczne, z wykładu.

**Metodologia ewaluacji**

Pierwszą metryką na jaką patrzyliśmy było accuracy, chcieliśmy, żeby model jak najczęściej zgadywał poprawny gatunek. Na drugim miejscu był recall per gatunek. 
Niektóre gatunki, np. rock i blues, są do siebie bardzo podobne i modele często je myliły, dlatego recall był dobrą metryką, która mówiła, który gatunek jest ciężki do zgadnięcia dla modelu.

**Szczegółowy opis uzyskanych wyników**

**Wnioski końcowe**

Problem klasyfikacji gatunku muzycznego był ciekawym i czasami trudnym tematem. 

Jedną z najtrudniejszych rzeczy było rozdzielenie gatunków.

# Pokaz

In [12]:
path = 'Data/Classicals.de - Tchaikovsky - Romeo and Juliet - Love Theme (Fantasy Overture).mp3'
ipd.Audio(path)

In [32]:
from FeatureEngineering.create_custom_dataset import create_custom_dataset
import joblib

paths = [
  'Data/Classicals.de - Tchaikovsky - Romeo and Juliet - Love Theme (Fantasy Overture).mp3',
  'Data/chappell-roan-good-luck-babe-official-lyric-video.mp3',
  'Data/ˏ` ssshhhiiittt! — танцы (www.lightaudio.ru).mp3',
]
df = create_custom_dataset([Path(p) for p in paths])

In [35]:
clf = joblib.load("prediction_model.joblib")
le = joblib.load("label_encoder.joblib")

X = df.drop(['genre', 'filename', 'length'], axis=1)

y_pred_encoded = clf.predict(X)
y_pred = le.inverse_transform(y_pred_encoded)

for p, pred in zip(paths, y_pred):
    print(f"{p} => {pred}")

Data/Classicals.de - Tchaikovsky - Romeo and Juliet - Love Theme (Fantasy Overture).mp3 => country
Data/chappell-roan-good-luck-babe-official-lyric-video.mp3 => pop
Data/ˏ` ssshhhiiittt! — танцы (www.lightaudio.ru).mp3 => hiphop
